In [9]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path 


DATA_HOT_SCORE = Path("data/hotscore")
OUTPUT_VOLATITLITY = Path("output/volatility")
OUTPUT_DATA = Path("output/data")

for p in (DATA_HOT_SCORE, OUTPUT_VOLATITLITY, OUTPUT_DATA):
    p.mkdir(parents=True, exist_ok=True)

In [10]:
def load_all_hotscore_files(directory=DATA_HOT_SCORE):
    files = sorted(glob.glob(str(directory / "hotscore_*.csv")))
    
    dfs = []
    
    for f in files:
        df = pd.read_csv(f)
        
        # extract date from filename
        date_str = os.path.basename(f).split("_")[1].replace(".csv", "")
        df["date"] = pd.to_datetime(date_str, format="%Y%m%d")
        
        

        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

df = load_all_hotscore_files()

df = df.sort_values(["symbol", "date"])
df.head()

,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap,date
4,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10,2026-06-22
274,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10,2026-06-22
544,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10,2026-06-22
814,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10,2026-06-22
1084,AAL,0.566667,0.093023,16.125,4.302815,0.238109,70078392.0,0.790698,0.620155,0.317829,1.066484e+10,2026-06-22


In [4]:
df.shape

(9970, 12)

In [11]:
cols = [
    "symbol",
    "HotScore",
    "TrendScore",
    "regularMarketPrice",
    "marketCap",
    "VolatilityScore"
]

df_sorted = (
    df.dropna(subset=["VolatilityScore"])
      .sort_values("VolatilityScore", ascending=False)
      .drop_duplicates("symbol", keep="first")[cols]
)

df_sorted.shape

(300, 6)

In [55]:
import plotly.express as px
import os
 

fig = px.bar(
    df_sorted.sort_values(by=["regularMarketPrice"], ascending=True).head(40),
    x="HotScore", 
    y="symbol",
    orientation="h",
    color="HotScore",
    text="regularMarketPrice",
    hover_name="symbol",
    hover_data=["TrendScore","VolatilityScore"], 
    title="Top Stocks by VolatilityScore",
    color_continuous_scale="Ice",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 50 Stocks by VolatilityScore",
    xaxis_title="Hot Score",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1400,
    margin=dict(l=40, r=40, t=60, b=40)
)
    
fig.show()


chart_path = os.path.join(OUTPUT_VOLATITLITY, "bar_volatility.html")
fig.write_html(chart_path, include_plotlyjs="cdn")
 

In [56]:
fig = px.scatter(
    df_sorted.sort_values(by=["regularMarketPrice"], ascending=True).head(60),
    x='TrendScore',
    y='HotScore',
    size='regularMarketPrice',
    text='regularMarketPrice',
    color='VolatilityScore',
    hover_name='symbol',
    log_x=True,
    title='HotScore vs Market Cap',
    color_continuous_scale="ice",
)


fig.update_traces(textposition="top right", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 60 Volatility Score Stocks", 
    xaxis_title="Trend Score",
    yaxis_title="Hot Score",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=800,
    margin=dict(l=40, r=40, t=60, b=40)
)


fig.show()


chart_path = os.path.join(OUTPUT_VOLATITLITY, "scatter_volatility.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [ ]:
import yfinance as yf

exchange_map = {
    "NYQ": "NYSE",
    "NMS": "NASDAQ",
    "NGM": "NASDAQ",
    "NCM": "NASDAQ",
    "NAS": "NASDAQ",
    "PCX": "NYSE",
    "ASE": "AMEX",
    "BTS": "NASDAQ",
    "YHD": "NASDAQ",
}

In [25]:
tickers = (
    df.sort_values(by="regularMarketPrice", ascending=True)
      ["symbol"]
      .drop_duplicates()   # remove duplicate symbols
      .head(50) 
      .tolist()
)
    

In [26]:
rows = []

for ticker in tickers:
    try:
        info = yf.Ticker(ticker).get_info()

        exchange = exchange_map.get(info.get("exchange"), info.get("exchange"))

        rows.append({
            "ticker": ticker,
            "market": exchange
        })

    except Exception:
        rows.append({
            "ticker": ticker,
            "market": None
        })

market_df = pd.DataFrame(rows)

In [27]:

out_file = OUTPUT_DATA / f"trading_view.csv"
market_df.to_csv(out_file, index=False)